In [218]:
import os

import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader,Dataset
from torchvision import transforms,models
from sklearn.metrics import accuracy_score

In [219]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PneumoniaDataset(Dataset):
    def __init__(self,root_dir,transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label in ['NORMAL','PNEUMONIA']:
            class_dir = os.path.join(root_dir,label)
            for img_name in os.listdir(class_dir):
                self.image_paths.append(os.path.join(class_dir,img_name))
                self.labels.append(0 if label == 'NORMAL' else 1)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, item):
        img_path = self.image_paths[item]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[item]

        if self.transform:
            image = self.transform(image)
        return image,label

In [220]:
from torch.utils.data import WeightedRandomSampler
from collections import Counter

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

train_dataset = PneumoniaDataset(root_dir='C:\\Projects\\python\\data_sets\\chest_xray\\train',transform=transform)
test_dataset = PneumoniaDataset(root_dir='C:\\Projects\\python\\data_sets\\chest_xray\\test',transform=transform)
val_dataset = PneumoniaDataset(root_dir='C:\\Projects\\python\\data_sets\\chest_xray\\val',transform=transform)

########
class_counts = Counter(train_dataset.labels)
class_weights = {0: 1.0/class_counts[0], 1: 1.0/class_counts[1]}
sample_weights = [class_weights[label] for label in train_dataset.labels]

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)

####train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False)

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features,2)#NORMAL,PNEUMONIA
model = model.to(device)

In [221]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [222]:
num_epochs = 15

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images,labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs,labels)
        loss.backward()

        optimizer.step()
        running_loss += loss

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}')

Epoch 1/15, Loss: 0.14410485327243805
Epoch 2/15, Loss: 0.08088823407888412
Epoch 3/15, Loss: 0.07364267110824585
Epoch 4/15, Loss: 0.06095802038908005
Epoch 5/15, Loss: 0.06179839372634888
Epoch 6/15, Loss: 0.06061934307217598
Epoch 7/15, Loss: 0.046837158501148224
Epoch 8/15, Loss: 0.05043969303369522
Epoch 9/15, Loss: 0.046794697642326355
Epoch 10/15, Loss: 0.03243632614612579
Epoch 11/15, Loss: 0.040263962000608444
Epoch 12/15, Loss: 0.034241288900375366
Epoch 13/15, Loss: 0.04160007834434509
Epoch 14/15, Loss: 0.033760447055101395
Epoch 15/15, Loss: 0.02356625534594059


In [223]:
model.eval()
val_labels = []
val_preds = []

with torch.no_grad():
    for images,labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _,preds = torch.max(outputs,1)

        val_labels.extend(labels.cpu().numpy())
        val_preds.extend(preds.cpu().numpy())

    val_accuracy = accuracy_score(val_labels,val_preds)
    print('Validation accuracy:',val_accuracy)

Validation accuracy: 0.9375


In [224]:
model.eval()
test_labels = []
test_preds = []


with torch.no_grad():
    for images,labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _,preds = torch.max(outputs,1)

        test_labels.extend(labels.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())

    test_accuracy = accuracy_score(test_labels,test_preds)
    print('Test accuracy:',test_accuracy)

Test accuracy: 0.8060897435897436


In [225]:
torch.save(model.state_dict(),'pneumonia_classifier.pth')

In [249]:
from PIL import Image
from torchvision import transforms
import torch

img_path = "C:\\Users\\jorda\\Downloads\\n2.jpg"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])

image = Image.open(img_path).convert('RGB')
image = transform(image).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model(image)
    _, predicted = torch.max(output, 1)
    print("Predicted Label:", predicted.item())


Predicted Label: 1


In [227]:
from collections import Counter

print("Train:", Counter(train_dataset.labels))
print("Val:", Counter(val_dataset.labels))
print("Test:", Counter(test_dataset.labels))


Train: Counter({1: 3875, 0: 1341})
Val: Counter({0: 8, 1: 8})
Test: Counter({1: 390, 0: 234})
